In [45]:
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import timezone, datetime
from sqlalchemy import create_engine, text
import urllib
import os
from dotenv import load_dotenv
import json

In [46]:
ROOT_DIR     = Path("..").resolve()
SILVER_DIR   = ROOT_DIR / "data" / "silver"
GOLD_DIR     = ROOT_DIR / "data" / "gold"
GOLD_DIR.mkdir(parents=True, exist_ok=True)
UTILS_DIR    = ROOT_DIR / "data" / "utils"

WC_FILE      = SILVER_DIR / "silver_wc_output.csv"
SOCIO_FILE   = SILVER_DIR / "valid_socioeconomics.csv"
MAPPING_FILE = UTILS_DIR / "country_confederations.json"

print(f"WC file   : {WC_FILE.name}")
print(f"Socio file: {SOCIO_FILE.name}")

WC file   : silver_wc_output.csv
Socio file: valid_socioeconomics.csv


In [48]:
# ── SQL Server connection ──────────────────────────────────────────────
# Credentials loaded from .env — never hardcoded
load_dotenv()

SERVER   = "sebastien-sql.database.windows.net"
DATABASE = os.getenv("SQL_DATABASE")  
USERNAME = "sqladmin"
PASSWORD = os.getenv("SQL_PASSWORD")  

conn_str = (
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={SERVER};"
    f"DATABASE={DATABASE};"
    f"UID={USERNAME};"
    f"PWD={PASSWORD};"
    "Encrypt=yes;"
    "TrustServerCertificate=yes;"
)
params = urllib.parse.quote_plus(conn_str)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# Test connection
with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
print("Connection OK")

Connection OK


In [8]:
wc = pd.read_csv(WC_FILE)
socio = pd.read_csv(SOCIO_FILE)

In [9]:
dim_date = (
    wc[["year"]]
    .drop_duplicates()
    .sort_values("year")
    .reset_index(drop=True)
)

dim_date["year_id"] = dim_date.index + 1

dim_date = dim_date[
    ["year_id", "year"]
]

display(dim_date)

,year_id,year
0,1,1998
1,2,2002
2,3,2006
3,4,2010
4,5,2014
5,6,2018


In [13]:
dim_date.to_csv( GOLD_DIR  / "dim_date.csv", index=False)

In [10]:
print(wc.columns.tolist())

['year', 'country', 'city', 'stage', 'home_team', 'away_team', 'home_score', 'away_score', 'winning_team', 'losing_team', 'date_clean', 'source_name', 'load_timestamp_clean', 'run_id']


In [23]:
tournament_names = {
    1998: "1998 France",
    2002: "2002 South Korea and Japan",
    2006: "2006 Germany",
    2010: "2010 South Africa",
    2014: "2014 Brazil",
    2018: "2018 Russia"
}

In [26]:
dim_tournament = (
    wc.groupby("year")
    .agg(
        num_matches=("year", "count")
    )
    .reset_index()
)

# Count number of teams per year
dim_tournament["num_teams"] = dim_tournament["year"].apply(
    lambda y: pd.concat([
        wc.loc[wc["year"] == y, "home_team"],
        wc.loc[wc["year"] == y, "away_team"]
    ]).nunique()
)

# Surrogate key
dim_tournament["tournament_id"] = dim_tournament.index + 1

# Bruk mapping til å lage riktig navn
dim_tournament["tournament_name"] = dim_tournament["year"].map(tournament_names)

# Riktig kolonnerekkefølge
dim_tournament = dim_tournament[
    ["tournament_id", "tournament_name", "num_matches", "num_teams"]
]

display(dim_tournament)

,tournament_id,tournament_name,num_matches,num_teams
0,1,1998 France,64,32
1,2,2002 South Korea and Japan,64,32
2,3,2006 Germany,64,32
3,4,2010 South Africa,64,32
4,5,2014 Brazil,64,32
5,6,2018 Russia,64,32


In [43]:
dim_tournament.to_csv( GOLD_DIR  / "dim_tournament.csv", index=False)

In [39]:
with open(MAPPING_FILE, "r") as f:
    confederation_mapping = json.load(f)

In [40]:
countries = pd.concat([
    wc["home_team"],
    wc["away_team"]
])

# Lag dim_country
dim_country = (
    pd.DataFrame({"country_name": countries})
    .drop_duplicates()
    .sort_values("country_name")
    .reset_index(drop=True)
)

# Legg til confederation fra JSON
dim_country["confederation"] = dim_country["country_name"].map(confederation_mapping)

# Lag surrogate key
dim_country["country_id"] = dim_country.index + 1

# Riktig kolonnerekkefølge
dim_country = dim_country[
    ["country_id", "country_name", "confederation"]
]

display(dim_country)

,country_id,country_name,confederation
0,1,Algeria,CAF
1,2,Angola,CAF
2,3,Argentina,CONMEBOL
3,4,Australia,AFC
4,5,Austria,UEFA
...,...,...,...
59,60,Tunisia,CAF
60,61,Turkey,UEFA
61,62,Ukraine,UEFA
62,63,United States,CONCACAF


In [ ]:
#Check to see if any are missing confederations
country_check = dim_country[dim_country["confederation"].isna()]
country_check

,country_id,country_name,confederation


In [42]:
dim_country.to_csv( GOLD_DIR  / "dim_country.csv", index=False)

Transfer to SQL

In [49]:
dim_date.to_sql(
    "dim_date",
    engine,
    if_exists="replace",
    index=False
)

dim_country.to_sql(
    "dim_country",
    engine,
    if_exists="replace",
    index=False
)

dim_tournament.to_sql(
    "dim_tournament",
    engine,
    if_exists="replace",
    index=False
)


6

In [54]:
test = pd.read_sql("SELECT TOP 10 * FROM dim_date", engine)
display(test)

,year_id,year
0,1,1998
1,2,2002
2,3,2006
3,4,2010
4,5,2014
5,6,2018
